# Import Library

In [25]:
import pandas as pd
from pathlib import Path

# Load Data

Baca file raw

In [ ]:
file_path = "./data/eur_idr_history_data.csv"
text = Path(file_path).read_text(encoding="utf-8-sig")

Bersihkan quote luar tiap baris

In [27]:
clean_lines = []

for line in text.splitlines():
    line = line.strip()
    
    if line.startswith('"') and line.endswith('"'):
        line = line[1:-1]
    
    line = line.replace('""', '"')
    clean_lines.append(line)

clean_text = "\n".join(clean_lines)


Load clean data

In [28]:
# 3. Load ulang sebagai CSV normal
df = pd.read_csv(file_path)

df.head()

,Date,Price,Open,High,Low,Vol.,Change %
0,06/01/2026,"20,739.0","20,739.0","20,739.0","20,739.0",NaN,-0.47%
1,05/29/2026,"20,836.4","20,716.4","20,884.7","20,701.3",NaN,0.58%
2,05/28/2026,"20,716.4","20,671.0","20,734.2","20,599.0",NaN,0.22%
3,05/27/2026,"20,671.0","20,678.1","20,734.2","20,663.9",NaN,-0.03%
4,05/26/2026,"20,678.1","20,651.5","20,716.5","20,625.8",NaN,0.13%


In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86 entries, 0 to 85
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      86 non-null     object 
 1   Price     86 non-null     object 
 2   Open      86 non-null     object 
 3   High      86 non-null     object 
 4   Low       86 non-null     object 
 5   Vol.      0 non-null      float64
 6   Change %  86 non-null     object 
dtypes: float64(1), object(6)
memory usage: 4.8+ KB


# Data preprocessing

## Rename column

In [30]:
df = df.rename(columns=
{
    "Date": "timestamp",
    "Price": "close",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Vol.": "volume"
})

In [31]:
print(df.head())
print(df.info())

    timestamp     close      open      high       low  volume Change %
0  06/01/2026  20,739.0  20,739.0  20,739.0  20,739.0     NaN   -0.47%
1  05/29/2026  20,836.4  20,716.4  20,884.7  20,701.3     NaN    0.58%
2  05/28/2026  20,716.4  20,671.0  20,734.2  20,599.0     NaN    0.22%
3  05/27/2026  20,671.0  20,678.1  20,734.2  20,663.9     NaN   -0.03%
4  05/26/2026  20,678.1  20,651.5  20,716.5  20,625.8     NaN    0.13%
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86 entries, 0 to 85
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   timestamp  86 non-null     object 
 1   close      86 non-null     object 
 2   open       86 non-null     object 
 3   high       86 non-null     object 
 4   low        86 non-null     object 
 5   volume     0 non-null      float64
 6   Change %   86 non-null     object 
dtypes: float64(1), object(6)
memory usage: 4.8+ KB
None


## Bersihkan angka ribuan

In [32]:
price_cols = ["open", "high", "low", "close"]
for col in price_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

## Convert timestamp

In [33]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86 entries, 0 to 85
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  86 non-null     datetime64[ns]
 1   close      86 non-null     float64       
 2   open       86 non-null     float64       
 3   high       86 non-null     float64       
 4   low        86 non-null     float64       
 5   volume     0 non-null      float64       
 6   Change %   86 non-null     object        
dtypes: datetime64[ns](1), float64(5), object(1)
memory usage: 4.8+ KB


## Isi dengan dummy data

In [35]:
# Volume tidak tersedia, isi dummy
df["volume"] = 0.0
df["amount"] = 0.0

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86 entries, 0 to 85
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  86 non-null     datetime64[ns]
 1   close      86 non-null     float64       
 2   open       86 non-null     float64       
 3   high       86 non-null     float64       
 4   low        86 non-null     float64       
 5   volume     86 non-null     float64       
 6   Change %   86 non-null     object        
 7   amount     86 non-null     float64       
dtypes: datetime64[ns](1), float64(6), object(1)
memory usage: 5.5+ KB


## Sort lama ke baru

In [37]:
df = df.sort_values("timestamp").reset_index(drop=True)

In [38]:
df.head()

,timestamp,close,open,high,low,volume,Change %,amount
0,2026-02-02,19797.1,19901.1,19951.0,19771.9,0.0,-0.47%,0.0
1,2026-02-03,19808.6,19797.1,19838.2,19743.3,0.0,0.06%,0.0
2,2026-02-04,19800.3,19808.6,19859.5,19771.8,0.0,-0.04%,0.0
3,2026-02-05,19820.7,19801.2,19896.4,19786.9,0.0,0.10%,0.0
4,2026-02-06,19929.4,19822.4,19945.4,19800.5,0.0,0.55%,0.0


## Ambil kolom untuk Kronos

In [39]:
kronos_df = df[["timestamp", "open", "high", "low", "close", "volume", "amount"]].copy()

In [40]:
kronos_df.head()

,timestamp,open,high,low,close,volume,amount
0,2026-02-02,19901.1,19951.0,19771.9,19797.1,0.0,0.0
1,2026-02-03,19797.1,19838.2,19743.3,19808.6,0.0,0.0
2,2026-02-04,19808.6,19859.5,19771.8,19800.3,0.0,0.0
3,2026-02-05,19801.2,19896.4,19786.9,19820.7,0.0,0.0
4,2026-02-06,19822.4,19945.4,19800.5,19929.4,0.0,0.0


## Hapus missing value kalau ada

In [41]:
kronos_df = kronos_df.dropna()
kronos_df.head()

,timestamp,open,high,low,close,volume,amount
0,2026-02-02,19901.1,19951.0,19771.9,19797.1,0.0,0.0
1,2026-02-03,19797.1,19838.2,19743.3,19808.6,0.0,0.0
2,2026-02-04,19808.6,19859.5,19771.8,19800.3,0.0,0.0
3,2026-02-05,19801.2,19896.4,19786.9,19820.7,0.0,0.0
4,2026-02-06,19822.4,19945.4,19800.5,19929.4,0.0,0.0


In [ ]:
kronos_df.to_csv("./data/eur_idr_kronos_clean.csv", index=False)

## Ambil kolom untuk Chronos-2

In [19]:
chronos_df = kronos_df[["timestamp", "close"]].copy()
chronos_df = chronos_df.rename(columns={"close": "value"})

chronos_df.head()

,timestamp,value
0,2021-01-01,17246.3
1,2021-01-04,17006.3
2,2021-01-05,17091.4
3,2021-01-06,17107.1
4,2021-01-07,17043.0


In [ ]:
# chronos_df.to_csv("./data/eur_idr_chronos_clean.csv", index=False)